# Context Engineering – Hands-On Demo

## Objective
In this lab, we will learn how to:
- Structure context instead of dumping prompts
- Separate static, dynamic, and memory context
- Build a real-world, AI system

### Use Case
College Course AI for an EdTech platform

> Key idea: **Context is infrastructure, not text.**


## Problem Statement

We want to build an AI system that:
- Acts as a College Course agent
- Follows company refund policies strictly
- Responds politely and clearly
- Uses user-specific information to decide responses

This is **not a chatbot**.  
This is a **context-aware AI system**.

## The Wrong Way: Context Dumping

Many people put everything into one prompt.
This approach does not scale and is hard to control.

In [8]:
bad_prompt = """
You are a college course assistant.
Help students with their courses.
Answer their questions and tell them what they need to do.
Here is some course information...
The student needs help.
"""
print(bad_prompt)


You are a college course assistant.
Help students with their courses.
Answer their questions and tell them what they need to do.
Here is some course information...
The student needs help.



## Context Engineering Approach

We break context into structured parts:

1. Static Context → Who the AI is and how it behaves
2. External Context → Policies or documents
3. Dynamic Context → User input and task data
4. Memory Context → What the system remembers

We assemble these intentionally.

## Step 1: Static Context

Static context defines:
- The role of the AI
- Rules and constraints
- Tone and behavior

This rarely changes.

In [9]:
SYSTEM_CONTEXT = """
You are a college course assistant for an EdTech platform.

Rules:
- Help students with their courses
- Answer questions
- Be polite
- Give useful information
- Follow the rules
- Escalate when necessary
"""

## Step 2: External Context (Policies)

The AI should not guess policies.
We explicitly provide them as context.

In [10]:
REFUND_POLICY = """
Refund Policy:
- Students can get refunds for courses
- Refunds are available sometimes
- Course progress may affect eligibility
- Subscriptions may or may not be refundable
- Contact support if you have questions
"""

## Step 3: Dynamic Context

Dynamic context changes per request.
This includes user input and user-specific data.

In [11]:
user_query = "I enrolled in the Python course 10 days ago and want to know if I can cancel my enrollment."

user_profile = {
    "role": "student",
    "course_name": "Python Programming",
    "course_progress": "15%",
    "enrollment_days_ago": 10
}

## Step 4: Context Assembly

We now assemble the context carefully.
Order and clarity matter.
This is **context engineering**.

In [13]:
final_prompt = f"""
{SYSTEM_CONTEXT}

Company Policy:
{REFUND_POLICY}

User Profile:
- Role: {user_profile['role']}
- Course Progress: {user_profile['course_progress']}
- Purchased: {user_profile['enrollment_days_ago']} days ago

User Question:
{user_query}
"""

In [14]:
print(final_prompt)



You are a college course assistant for an EdTech platform.

Rules:
- Help students with their courses
- Answer questions
- Be polite
- Give useful information
- Follow the rules
- Escalate when necessary


Company Policy:

Refund Policy:
- Students can get refunds for courses
- Refunds are available sometimes
- Course progress may affect eligibility
- Subscriptions may or may not be refundable
- Contact support if you have questions


User Profile:
- Role: student
- Course Progress: 15%
- Purchased: 10 days ago

User Question:
I enrolled in the Python course 10 days ago and want to know if I can cancel my enrollment.



In [15]:
from google.colab import userdata

# Access the API key stored in Colab Secrets
MY_API_KEY = userdata.get('api_key')

# Now, use this variable when initializing your OpenAI client
# client = OpenAI(api_key=MY_API_KEY, base_url="https://apidev.navigatelabsai.com")

print("API key loaded successfully from Colab Secrets.")
print("Please update the client initialization in the cell above (hok-nnOpgi2r) to use `api_key=MY_API_KEY`.")

API key loaded successfully from Colab Secrets.
Please update the client initialization in the cell above (hok-nnOpgi2r) to use `api_key=MY_API_KEY`.


## Step 5: Call the Model

We now send the structured context to the model.

In [17]:
from openai import OpenAI

client = OpenAI(api_key=MY_API_KEY, base_url="https://nexusapi.navigatelabs.ai")

response = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {"role": "system", "content": SYSTEM_CONTEXT},
        {"role": "user", "content": final_prompt}
    ]
)

print(response.choices[0].message.content)

Hi there! Thanks for reaching out.

Regarding your question about canceling your enrollment in the Python course, our company policy states that refunds are sometimes available and course progress can be a factor in eligibility. You mentioned you enrolled 10 days ago and have completed 15% of the course.

To get a definitive answer on your eligibility and to proceed with a cancellation, please reach out directly to our support team. They will be able to review your specific situation and provide you with all the necessary information.


## Why Did the AI Respond Correctly?

The model:

* Did not make an unsupported promise about a refund or cancellation
* Acknowledged the user's request and explained that eligibility may depend on enrollment details
* Followed the policy by directing the user to the support team for confirmation
* Maintained a polite and professional tone

This happened **because of the context provided to the model**, not simply because the model is smart.

## Step 6: Memory Context

Real AI systems remember important user information.
We store **summarized memory**, not full chat history.

In [21]:
SESSION_MEMORY = """
User enrolled in the Python course 10 days ago.
User wants to know whether they can cancel their enrollment.
"""

## Context Assembly with Memory

We now include session memory into the context.

In [22]:
final_prompt_with_memory = f"""
{SYSTEM_CONTEXT}

Session Memory:
{SESSION_MEMORY}

Company Policy:
{REFUND_POLICY}

User Question:
{user_query}
"""

In [23]:
response = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {"role": "system", "content": SYSTEM_CONTEXT},
        {"role": "user", "content": final_prompt_with_memory}
    ]
)

print(response.choices[0].message.content)

Hello there! Thanks for reaching out with your question.

Regarding your enrollment in the Python course, I understand you're looking to cancel it. Our refund policy allows for cancellations and refunds in some situations, and eligibility can sometimes depend on factors like how much of the course you've completed or the type of enrollment you have.

To get the most accurate information specific to your situation and to discuss cancellation options, the best approach is to contact our support team directly. They will be able to review your enrollment details and guide you through the process.


## Your Task

1. Choose a real-world domain
2. Define:
   - Static context
   - Dynamic context
   - Memory context
3. Assemble context in code
4. Show one real workflow
5. Explain how this scales to your capstone

No generic chatbots allowed.